# 19. Stacking Robustness Ablation

**Tujuan:** Robustness Ablation pada Stacking Ensemble: C1(Full) / C2(Top-15) / C3(Top-10) / C4(Top-5).
Apakah sweet spot bergeser dibanding single model? Apakah ensemble lebih toleran di Top-5?

**Input:** `cleaned_100.pkl`, `stacking_baseline_15.pkl`, `stacking_adversarial_17.pkl`, `robustness_ablation_07.pkl`

**Output:** `stacking_robustness_19.pkl`, PNG comparison

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
TEST_SIZE = 0.20
DATA_DIR = '../data/'
EPSILON = 0.1

print('Libraries loaded.')

## 1. Load Data & Feature Ranking

In [ ]:
with open(os.path.join(DATA_DIR, 'cleaned_100.pkl'), 'rb') as f:
    data = pickle.load(f)
X = data['X']
y = data['y']

with open(os.path.join(DATA_DIR, 'stacking_baseline_15.pkl'), 'rb') as f:
    stack_data = pickle.load(f)
n_classes = stack_data['n_classes']

# Feature ranking
xgb_model = stack_data['base_models']['XGBoost']
importances = xgb_model.feature_importances_
ranked_idx = np.argsort(importances)[::-1]

# Load single model robustness for comparison
with open(os.path.join(DATA_DIR, 'robustness_ablation_07.pkl'), 'rb') as f:
    single_robustness = pickle.load(f)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
print(f'Data loaded. Train={X_train.shape[0]}, Test={X_test.shape[0]}')

## 2. Robustness Evaluation Function

In [ ]:
def evaluate_robustness_stacking(X_tr, X_te, y_tr, y_te, n_classes, epsilon, config_name):
    """
    For a given feature subset:
    1. Train baseline stacking → evaluate on clean (S1) and adversarial (S2)
    2. Train robust stacking (augmented) → evaluate on clean (S3) and adversarial (S4)
    """
    def _train_stack(X_tr_in, y_tr_in, X_te_in):
        base_defs = [
            ('XGB', XGBClassifier(max_depth=6, n_estimators=100, learning_rate=0.1,
                                  use_label_encoder=False, eval_metric='mlogloss',
                                  random_state=RANDOM_SEED, verbosity=0)),
            ('LGBM', LGBMClassifier(max_depth=6, n_estimators=100, learning_rate=0.1,
                                    random_state=RANDOM_SEED, verbose=-1)),
            ('CAT', CatBoostClassifier(depth=6, iterations=100, learning_rate=0.1,
                                       random_seed=RANDOM_SEED, verbose=0))
        ]
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
        meta_tr = np.zeros((X_tr_in.shape[0], n_classes * 3))
        meta_te = np.zeros((X_te_in.shape[0], n_classes * 3))
        models = {}
        
        for idx, (nm, mdl) in enumerate(base_defs):
            oof = np.zeros((X_tr_in.shape[0], n_classes))
            te_p = np.zeros((X_te_in.shape[0], n_classes))
            for ti, vi in cv.split(X_tr_in, y_tr_in):
                m = mdl.__class__(**mdl.get_params())
                m.fit(X_tr_in[ti], y_tr_in[ti])
                oof[vi] = m.predict_proba(X_tr_in[vi])
                te_p += m.predict_proba(X_te_in) / cv.n_splits
            meta_tr[:, idx*n_classes:(idx+1)*n_classes] = oof
            meta_te[:, idx*n_classes:(idx+1)*n_classes] = te_p
            final_m = mdl.__class__(**mdl.get_params())
            final_m.fit(X_tr_in, y_tr_in)
            models[nm] = final_m
        
        scl = StandardScaler()
        lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, multi_class='multinomial')
        lr.fit(scl.fit_transform(meta_tr), y_tr_in)
        y_pred = lr.predict(scl.transform(meta_te))
        return models, lr, scl, y_pred
    
    def _predict(X_in, models, lr, scl):
        meta = np.zeros((X_in.shape[0], n_classes * 3))
        for idx, (nm, mdl) in enumerate(models.items()):
            meta[:, idx*n_classes:(idx+1)*n_classes] = mdl.predict_proba(X_in)
        return lr.predict(scl.transform(meta))
    
    # S1: Baseline + Clean
    models_b, lr_b, scl_b, _ = _train_stack(X_tr, y_tr, X_te)
    y_s1 = _predict(X_te, models_b, lr_b, scl_b)
    
    # Generate adversarial
    np.random.seed(RANDOM_SEED)
    noise = epsilon * np.random.choice([-1, 1], size=X_te.shape)
    X_adv = X_te + noise
    
    # S2: Baseline + Adversarial
    y_s2 = _predict(X_adv, models_b, lr_b, scl_b)
    
    # Augmented training
    n_aug = int(X_tr.shape[0] * 0.2)
    aug_noise = epsilon * np.random.choice([-1, 1], size=(n_aug, X_tr.shape[1]))
    X_aug = X_tr[:n_aug] + aug_noise
    X_robust = np.vstack([X_tr, X_aug])
    y_robust = np.concatenate([y_tr, y_tr[:n_aug]])
    
    # S3 & S4: Robust
    models_r, lr_r, scl_r, _ = _train_stack(X_robust, y_robust, X_te)
    y_s3 = _predict(X_te, models_r, lr_r, scl_r)
    y_s4 = _predict(X_adv, models_r, lr_r, scl_r)
    
    return {
        'config': config_name,
        'n_features': X_tr.shape[1],
        'mcc_s1': matthews_corrcoef(y_te, y_s1),
        'mcc_s2': matthews_corrcoef(y_te, y_s2),
        'mcc_s3': matthews_corrcoef(y_te, y_s3),
        'mcc_s4': matthews_corrcoef(y_te, y_s4),
        'security_gap': matthews_corrcoef(y_te, y_s1) - matthews_corrcoef(y_te, y_s2),
        'recovery': matthews_corrcoef(y_te, y_s4) - matthews_corrcoef(y_te, y_s2)
    }

print('Evaluation function defined.')

## 3. Run Robustness Ablation (C1-C4)

In [ ]:
configs = {
    'C1 (Full)': ranked_idx.tolist(),
    'C2 (Top-15)': ranked_idx[:15].tolist(),
    'C3 (Top-10)': ranked_idx[:10].tolist(),
    'C4 (Top-5)': ranked_idx[:5].tolist(),
}

robustness_results = []

print('='*70)
print('  STACKING ROBUSTNESS ABLATION STUDY')
print('='*70)

for cfg_name, feat_idx in configs.items():
    print(f'\n  {cfg_name} ({len(feat_idx)} features)...')
    X_tr_sub = X_train[:, feat_idx]
    X_te_sub = X_test[:, feat_idx]
    
    result = evaluate_robustness_stacking(
        X_tr_sub, X_te_sub, y_train, y_test, n_classes, EPSILON, cfg_name
    )
    robustness_results.append(result)
    
    print(f'    S1={result["mcc_s1"]:.4f} | S2={result["mcc_s2"]:.4f} | '
          f'S3={result["mcc_s3"]:.4f} | S4={result["mcc_s4"]:.4f}')
    print(f'    Gap={result["security_gap"]:.4f} | Recovery={result["recovery"]:.4f}')

rob_df = pd.DataFrame(robustness_results)
print('\n' + rob_df.to_string(index=False))

## 4. Comparison: Single vs Stacking Robustness

In [ ]:
print('\n' + '='*70)
print('  COMPARISON: Single XGBoost vs Stacking (Security Gap & Recovery)')
print('='*70)
print(f'{"Config":<12} | {"Single Gap":<12} | {"Stack Gap":<12} | {"Single Recov":<14} | {"Stack Recov":<12}')
print('-'*65)

for res in robustness_results:
    cfg = res['config']
    s_gap = res['security_gap']
    s_rec = res['recovery']
    # Placeholder for single model (from notebook 07 pkl)
    print(f'{cfg:<12} | {"N/A":<12} | {s_gap:<12.4f} | {"N/A":<14} | {s_rec:<12.4f}')

## 5. Visualisasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs_labels = [r['config'] for r in robustness_results]
x = np.arange(len(configs_labels))

# MCC S4 per config
mcc_s4_vals = [r['mcc_s4'] for r in robustness_results]
axes[0].bar(x, mcc_s4_vals, color='darkorange', alpha=0.8)
axes[0].set_ylabel('MCC (S4 - Robust+Adversarial)')
axes[0].set_title('Stacking Robustness per Feature Config')
axes[0].set_xticks(x)
axes[0].set_xticklabels(configs_labels, fontsize=9)
axes[0].set_ylim(0, 1.1)
axes[0].grid(axis='y', alpha=0.3)

# Security Gap vs Recovery
gaps = [r['security_gap'] for r in robustness_results]
recoveries = [r['recovery'] for r in robustness_results]
width = 0.35
axes[1].bar(x - width/2, gaps, width, label='Security Gap', color='red', alpha=0.6)
axes[1].bar(x + width/2, recoveries, width, label='Recovery', color='green', alpha=0.6)
axes[1].set_ylabel('MCC Difference')
axes[1].set_title('Security Gap vs Recovery (Stacking)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(configs_labels, fontsize=9)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'stacking_robustness_ablation.png'), bbox_inches='tight')
plt.show()
print('Saved: stacking_robustness_ablation.png')

## 6. Save Results

In [ ]:
output = {
    'robustness_results': robustness_results,
    'robustness_df': rob_df,
    'configs': configs,
    'epsilon': EPSILON
}

with open(os.path.join(DATA_DIR, 'stacking_robustness_19.pkl'), 'wb') as f:
    pickle.dump(output, f)

print('Saved: stacking_robustness_19.pkl')
print('\nNotebook 19 selesai. Lanjut ke 20 (Final Evaluation).')